# Attempt exclusion and explicit restart tests

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import unittest,tempfile
from pathlib import Path
from contract import reserve_attempt,write_json
class Checks(unittest.TestCase):
    def test_new_attempt_and_no_implicit_resume(self):
        with tempfile.TemporaryDirectory() as t:
            root=Path(t)/'P1';out=reserve_attempt(root,'attempt-01');(out/'sentinel').write_text('original')
            with self.assertRaises(FileExistsError):reserve_attempt(root,'attempt-01')
            with self.assertRaises(FileExistsError):reserve_attempt(root,'attempt-02')
            self.assertEqual((out/'sentinel').read_text(),'original')
    def test_completed_run_rejected_even_with_new_attempt(self):
        with tempfile.TemporaryDirectory() as t:
            root=Path(t)/'P1';out=reserve_attempt(root,'attempt-01');write_json(out/'result.json',{'status':'PASS'});write_json(root/'COMPLETE.json',{'done':True})
            with self.assertRaises(FileExistsError):reserve_attempt(root,'attempt-02',restart_of='attempt-01')
            self.assertFalse((root/'attempt-02').exists())
    def test_only_one_explicit_failed_restart(self):
        with tempfile.TemporaryDirectory() as t:
            root=Path(t)/'P1';a=reserve_attempt(root,'attempt-01');write_json(a/'result.json',{'status':'FAIL'})
            b=reserve_attempt(root,'attempt-02','attempt-01');write_json(b/'result.json',{'status':'FAIL'})
            with self.assertRaises(ValueError):reserve_attempt(root,'attempt-03','attempt-02')
    def test_incomplete_attempt_cannot_be_resumed_as_failure(self):
        with tempfile.TemporaryDirectory() as t:
            root=Path(t)/'P1';reserve_attempt(root,'attempt-01')
            with self.assertRaises(FileNotFoundError):reserve_attempt(root,'attempt-02','attempt-01')
    def test_escaping_attempt_rejected(self):
        with tempfile.TemporaryDirectory() as t:
            with self.assertRaises(ValueError):reserve_attempt(Path(t)/'run','../escape')
print('Attempt exclusion and explicit restart tests definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Attempt exclusion and explicit restart tests definitions/execution completed.


In [3]:
suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print('Tests run:',result.testsRun)

test_completed_run_rejected_even_with_new_attempt (__main__.Checks.test_completed_run_rejected_even_with_new_attempt) ... 

ok


test_escaping_attempt_rejected (__main__.Checks.test_escaping_attempt_rejected) ... 

ok


test_incomplete_attempt_cannot_be_resumed_as_failure (__main__.Checks.test_incomplete_attempt_cannot_be_resumed_as_failure) ... 

...

ok


test_only_one_explicit_failed_restart (__main__.Checks.test_only_one_explicit_failed_restart) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 0.011s

OK


Tests run: 5
